# Federal Funds Target Rate - FOMC Meeting Data Processing

**Goal:**
1. Process the `Federal Funds Target Range Upper Limit.csv` data.
2. Incorporate explicit FOMC meeting dates to identify decision days.
3. Create labels (`higher`, `same`, `lower`) reflecting the decision made at each meeting compared to the previous meeting.

Source: https://fred.stlouisfed.org/series/FEDFUNDS

In [ ]:
import pandas as pd
import numpy as np

# 1. Load the continuous daily data
file_path = "https://raw.githubusercontent.com/echochoho1010/forecasting_fed_rate/master/data/Federal%20Funds%20Target%20Range%20Upper%20Limit.csv"
df = pd.read_csv(file_path)

# Clean Data
df['DFEDTARU'] = pd.to_numeric(df['DFEDTARU'], errors='coerce')
df['observation_date'] = pd.to_datetime(df['observation_date'])
df = df.dropna(subset=['DFEDTARU']).sort_values('observation_date').reset_index(drop=True)

df.head()

### Incorporate FOMC Meeting Dates

We use the provided list of past FOMC meeting dates. The new target rate takes effect on the meeting day itself or the day after. Because the FRED series is continuous, measuring the target rate exactly **1 day after** the meeting will securely yield the post-decision target rate for that meeting. 

Source: https://www.federalreserve.gov/monetarypolicy/fomccalendars.htm

In [2]:
raw_meetings = [
    '2009-01-28', '2009-03-18', '2009-04-29', '2009-06-24', '2009-08-12', '2009-09-23', '2009-11-04', '2009-12-16', 
    '2010-01-27', '2010-03-16', '2010-04-28', '2010-06-23', '2010-08-10', '2010-09-21', '2010-11-03', '2010-12-14', 
    '2011-01-26', '2011-03-15', '2011-04-27', '2011-06-22', '2011-08-09', '2011-09-21', '2011-11-02', '2011-12-13', 
    '2012-01-25', '2012-03-13', '2012-04-25', '2012-06-20', '2012-07-31', '2012-09-13', '2012-10-24', '2012-12-12', 
    '2013-01-30', '2013-03-20', '2013-05-01', '2013-06-19', '2013-07-31', '2013-09-18', '2013-10-30', '2013-12-18', 
    '2014-01-29', '2014-03-19', '2014-04-30', '2014-06-18', '2014-07-30', '2014-09-17', '2014-10-29', '2014-12-17', 
    '2015-01-28', '2015-03-18', '2015-04-29', '2015-06-17', '2015-07-29', '2015-09-17', '2015-10-28', '2015-12-16', 
    '2016-01-27', '2016-03-16', '2016-04-27', '2016-06-15', '2016-07-27', '2016-09-21', '2016-11-02', '2016-12-14', 
    '2017-02-01', '2017-03-15', '2017-05-03', '2017-06-14', '2017-07-26', '2017-09-20', '2017-11-01', '2017-12-13', 
    '2018-01-31', '2018-03-21', '2018-05-02', '2018-06-13', '2018-08-01', '2018-09-26', '2018-11-08', '2018-12-19', 
    '2019-01-30', '2019-03-19', '2019-05-01', '2019-06-19', '2019-07-31', '2019-09-18', '2019-10-04', '2019-10-30', '2019-12-11', 
    '2020-01-29', '2020-03-03', '2020-03-15', '2020-03-23', '2020-04-29', '2020-06-10', '2020-07-29', '2020-09-16', '2020-11-05', '2020-12-16', 
    '2021-01-27', '2021-03-17', '2021-04-28', '2021-06-16', '2021-07-28', '2021-09-22', '2021-11-03', '2021-12-15', 
    '2022-01-26', '2022-03-16', '2022-05-04', '2022-06-15', '2022-07-27', '2022-09-21', '2022-11-02', '2022-12-14', 
    '2023-02-01', '2023-03-22', '2023-05-03', '2023-06-14', '2023-07-26', '2023-09-20', '2023-11-01', '2023-12-13', 
    '2024-01-31', '2024-03-20', '2024-05-01', '2024-06-12', '2024-07-31', '2024-09-18', '2024-11-07', '2024-12-18', 
    '2025-01-28', '2025-03-19', '2025-05-07', '2025-06-18', '2025-07-30', '2025-09-17', '2025-10-29', '2025-12-10', 
    '2026-01-28'
]

# Ensure date formats are correct and drop any accidental duplicates (like 2015-04-29)
df_meetings = pd.DataFrame({'meeting_date': pd.to_datetime(raw_meetings)})
df_meetings = df_meetings.drop_duplicates().sort_values('meeting_date').reset_index(drop=True)

# The decision establishes the target rate that holds the day after the meeting
df_meetings['post_meeting_date'] = df_meetings['meeting_date'] + pd.Timedelta(days=1)

# Merge with the continuous target rate dataframe
df_decisions = pd.merge(
    df_meetings,
    df[['observation_date', 'DFEDTARU']],
    left_on='post_meeting_date',
    right_on='observation_date',
    how='left'
)

# Clean up columns and keep only relevant variables
df_decisions = df_decisions[['meeting_date', 'DFEDTARU']].rename(columns={'DFEDTARU': 'target_rate'})

print("Total meetings matched to rates:", len(df_decisions))
df_decisions.head()

Total meetings matched to rates: 140


,meeting_date,target_rate
0,2009-01-28,0.25
1,2009-03-18,0.25
2,2009-04-29,0.25
3,2009-06-24,0.25
4,2009-08-12,0.25


### Determine FOMC Action Labels (`higher`, `same`, `lower`)

Now we calculate the difference between the target rate produced by the *current* meeting and the target rate maintained from the *previous* meeting.

In [3]:
def get_action_label(diff_val):
    if pd.isna(diff_val):
        return None
    elif diff_val > 0:
        return 'higher'
    elif diff_val < 0:
        return 'lower'
    else:
        return 'same'

# Calculate diff from the immediately preceding meeting
df_decisions['previous_rate'] = df_decisions['target_rate'].shift(1)
df_decisions['rate_change'] = df_decisions['target_rate'] - df_decisions['previous_rate']
df_decisions['decision'] = df_decisions['rate_change'].apply(get_action_label)

display(df_decisions.head(15))

,meeting_date,target_rate,previous_rate,rate_change,decision
0,2009-01-28,0.25,NaN,NaN,NaN
1,2009-03-18,0.25,0.25,0.0,same
2,2009-04-29,0.25,0.25,0.0,same
3,2009-06-24,0.25,0.25,0.0,same
4,2009-08-12,0.25,0.25,0.0,same
5,2009-09-23,0.25,0.25,0.0,same
6,2009-11-04,0.25,0.25,0.0,same
7,2009-12-16,0.25,0.25,0.0,same
8,2010-01-27,0.25,0.25,0.0,same
9,2010-03-16,0.25,0.25,0.0,same


### Visualize Federal Funds Rate
Let's plot the target rate over time.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(figsize=(12, 6))
sns.lineplot(x='meeting_date', y='target_rate', data=df_decisions, ax=ax, color='steelblue', linewidth=1.5)
ax.scatter(df_decisions['meeting_date'], df_decisions['target_rate'],
           color='red', s=25, zorder=5, label='FOMC Meeting')
ax.set_title('Federal Funds Target Rate Over Time')
ax.set_xlabel('Date')
ax.set_ylabel('Target Rate (%)')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

### Explore Rate Policy Distribution

Let's see an overview of what the Federal Reserve actually did across these meetings.

In [5]:
print("Overall Rate Changes at Meetings (Distribution):")
display(df_decisions['decision'].value_counts())

print("\nSample of Meetings where rates CHANGED:")
display(df_decisions[df_decisions['decision'].isin(['higher', 'lower'])].head(10))

Overall Rate Changes at Meetings (Distribution):


decision
same      108
higher     20
lower      11
Name: count, dtype: int64


Sample of Meetings where rates CHANGED:


,meeting_date,target_rate,previous_rate,rate_change,decision
55,2015-12-16,0.50,0.25,0.25,higher
63,2016-12-14,0.75,0.50,0.25,higher
65,2017-03-15,1.00,0.75,0.25,higher
67,2017-06-14,1.25,1.00,0.25,higher
71,2017-12-13,1.50,1.25,0.25,higher
73,2018-03-21,1.75,1.50,0.25,higher
75,2018-06-13,2.00,1.75,0.25,higher
77,2018-09-26,2.25,2.00,0.25,higher
79,2018-12-19,2.50,2.25,0.25,higher
84,2019-07-31,2.25,2.50,-0.25,lower


---
### Incorporating Crude Oil Data
We now connect the crude oil data to the end of this process.

In [ ]:
# Load WTI Crude Oil Data
df_oil = pd.read_csv('https://raw.githubusercontent.com/echochoho1010/forecasting_fed_rate/master/data/WTI%20Crude%20Oil%20Prices.csv')
df_oil['observation_date'] = pd.to_datetime(df_oil.iloc[:,0], errors='coerce')
df_oil['oil_price'] = pd.to_numeric(df_oil.iloc[:, 1], errors='coerce')
df_oil = df_oil.dropna(subset=['observation_date', 'oil_price']).sort_values('observation_date').reset_index(drop=True)

oil_stats = []
for date in df_decisions['meeting_date']:
    start_date = date - pd.Timedelta(days=30)
    history_30d = df_oil[(df_oil['observation_date'] >= start_date) & (df_oil['observation_date'] < date)]
    if not history_30d.empty:
        avg_price = history_30d['oil_price'].mean()
        if history_30d['oil_price'].iloc[0] != 0:
            price_change_pct = (history_30d['oil_price'].iloc[-1] - history_30d['oil_price'].iloc[0]) / history_30d['oil_price'].iloc[0]
        else:
            price_change_pct = np.nan
    else:
        avg_price = np.nan
        price_change_pct = np.nan
    oil_stats.append({'meeting_date': date, 'oil_price_30d_avg': avg_price, 'oil_price_30d_pct_change': price_change_pct})

df_oil_stats = pd.DataFrame(oil_stats)
df_merged = pd.merge(df_decisions, df_oil_stats, on='meeting_date', how='inner').dropna()
display(df_merged.head())

### Visualize Normalized WTI Crude Oil Price

We normalize the WTI price to the [0, 1] range using min-max scaling so it can be compared directly against the normalized Fed Funds Rate on the same axis.

In [ ]:
# --- Chart 1: Normalized WTI Crude Oil Price (Daily) with FOMC Meeting Markers ---
oil_min = df_oil['oil_price'].min()
oil_max = df_oil['oil_price'].max()
df_oil['oil_price_normalized'] = (df_oil['oil_price'] - oil_min) / (oil_max - oil_min)

# For each FOMC meeting, find the closest available WTI price on or before that date
meeting_wti_rows = []
for date in df_decisions['meeting_date']:
    available = df_oil[df_oil['observation_date'] <= date]
    if not available.empty:
        meeting_wti_rows.append({
            'meeting_date': date,
            'oil_price_normalized': available.iloc[-1]['oil_price_normalized']
        })
df_meeting_wti = pd.DataFrame(meeting_wti_rows)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df_oil['observation_date'], df_oil['oil_price_normalized'],
        color='darkorange', linewidth=1.2, label='WTI Crude Oil (Normalized)')
ax.scatter(df_meeting_wti['meeting_date'], df_meeting_wti['oil_price_normalized'],
           color='red', s=25, zorder=5, label='FOMC Meeting')
ax.set_title('Normalized WTI Crude Oil Price Over Time\n(Min-Max Scaled to [0, 1])')
ax.set_xlabel('Date')
ax.set_ylabel('Normalized Price (0–1)')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- Chart 2: Combined Normalized Fed Rate + WTI (at FOMC Meeting Dates) ---
# Normalize target rate
rate_min = df_merged['target_rate'].min()
rate_max = df_merged['target_rate'].max()
df_merged['target_rate_normalized'] = (df_merged['target_rate'] - rate_min) / (rate_max - rate_min)

# Normalize WTI 30-day average (aligned to meeting dates)
oil_avg_min = df_merged['oil_price_30d_avg'].min()
oil_avg_max = df_merged['oil_price_30d_avg'].max()
df_merged['oil_price_30d_normalized'] = (df_merged['oil_price_30d_avg'] - oil_avg_min) / (oil_avg_max - oil_avg_min)

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(df_merged['meeting_date'], df_merged['target_rate_normalized'],
        color='steelblue', linewidth=2, label='Fed Funds Rate (Normalized)')
ax.plot(df_merged['meeting_date'], df_merged['oil_price_30d_normalized'],
        color='darkorange', linewidth=2, alpha=0.85,
        label='WTI Crude Oil – 30d Avg (Normalized)')

# Highlight meetings where the rate actually changed, with ↑ / ↓ annotations
changed = df_merged[df_merged['decision'].isin(['higher', 'lower'])]
ax.scatter(changed['meeting_date'], changed['target_rate_normalized'],
           color='red', s=60, zorder=6, label='Rate Change (FOMC)')

for _, row in changed.iterrows():
    symbol = '↑' if row['decision'] == 'higher' else '↓'
    ax.annotate(symbol,
                xy=(row['meeting_date'], row['target_rate_normalized']),
                xytext=(0, 8), textcoords='offset points',
                ha='center', fontsize=9, color='red', fontweight='bold')

ax.set_title('Normalized Federal Funds Rate vs. WTI Crude Oil Price\n(at FOMC Meeting Dates, Min-Max Scaled to [0, 1])')
ax.set_xlabel('Date')
ax.set_ylabel('Normalized Value (0–1)')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()